# Normalise selected datasets

Uses local `data/` pickles from `geo_data_download.py` and auto-selection from `select_datasets.py`.
No MethylGPT probe list.

In [ ]:
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from inmoose.pycombat import pycombat_norm

ROOT = Path.cwd().resolve()
if ROOT.name == "01_data_download":
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "01_data_download")]

from shared.paths import GSE_LIST, PROCESSED_DIR
from select_datasets import get_selected_datasets

ALL_DATASETS = get_selected_datasets(require_age=True) or list(GSE_LIST)
FINAL_DIR = ROOT / "data" / "03_final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
print("datasets:", ALL_DATASETS)

cpg_list, meta_list, available = [], [], []
best_i, max_n = 0, 0
for name in ALL_DATASETS:
    path = PROCESSED_DIR / name / "methylation.pkl"
    if not path.exists():
        print("SKIP missing", path)
        continue
    with open(path, "rb") as f:
        obj = pickle.load(f)
    obj["meta"]["Dataset"] = name
    available.append(name)
    cpg_list.append(obj["CpG"])
    meta_list.append(obj["meta"])
    if obj["CpG"].shape[1] > max_n:
        max_n, best_i = obj["CpG"].shape[1], len(cpg_list) - 1

if not cpg_list:
    raise FileNotFoundError(f"No pickles in {PROCESSED_DIR}. Run geo_data_download.py")

ALL_DATASETS = available
shared = cpg_list[0].index
for df in cpg_list[1:]:
    shared = shared.intersection(df.index)
cpg_list = [df.reindex(shared) for df in cpg_list]
print("reference:", ALL_DATASETS[best_i], "shared CpGs:", len(shared))

def plot_pca(X, meta, title):
    X = X.fillna(X.median()).dropna()
    if X.empty:
        print("PCA skipped:", title)
        return
    pcs = PCA(2).fit_transform(X.T)
    fig, ax = plt.subplots(figsize=(8, 6))
    for ds, color in zip(meta["Dataset"].unique(), cm.tab10.colors):
        m = meta["Dataset"] == ds
        ax.scatter(pcs[m, 0], pcs[m, 1], label=ds, color=color, alpha=0.7)
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.show()

plot_pca(pd.concat(cpg_list, axis=1), pd.concat(meta_list), "PCA pre-correction")

ref_cpg, ref_meta = cpg_list[best_i].copy(), meta_list[best_i].copy()
store = {ALL_DATASETS[best_i]: {"CpG": ref_cpg, "meta": ref_meta}}

for i, (cpg, meta) in enumerate(zip(cpg_list, meta_list)):
    if i == best_i:
        continue
    if "age" not in ref_meta.columns or "age" not in meta.columns:
        print("SKIP (no age):", ALL_DATASETS[i])
        continue
    ok = list(set(ref_cpg.index[~ref_cpg.isna().any(axis=1)]) & set(cpg.index[~cpg.isna().any(axis=1)]))
    combined = pd.concat([ref_cpg.loc[ok], cpg.loc[ok]], axis=1)
    batch = [0] * ref_cpg.shape[1] + [1] * cpg.shape[1]
    cov = pd.DataFrame(
        {"age": np.concatenate([ref_meta["age"].values, meta["age"].values])},
        index=combined.columns,
    )
    corr = pycombat_norm(combined, batch, ref_batch=0, covar_mod=cov, na_cov_action="remove")
    out = corr.iloc[:, ref_cpg.shape[1]:]
    full = pd.DataFrame(np.nan, index=ref_cpg.index, columns=out.columns)
    full.loc[ok] = out.values
    store[ALL_DATASETS[i]] = {"CpG": full, "meta": meta[meta["ID"].isin(full.columns)]}

for name, data in store.items():
    out = FINAL_DIR / name
    out.mkdir(parents=True, exist_ok=True)
    with open(out / "methylation.pkl", "wb") as f:
        pickle.dump(data, f)
    print("saved", name, data["CpG"].shape)

comb_cpg = pd.concat([store[d]["CpG"] for d in store], axis=1)
comb_meta = pd.concat([store[d]["meta"] for d in store])
plot_pca(comb_cpg, comb_meta, "PCA post-correction")
out = PROCESSED_DIR / "combined"
out.mkdir(parents=True, exist_ok=True)
with open(out / "methylation.pkl", "wb") as f:
    pickle.dump({"CpG": comb_cpg, "meta": comb_meta}, f)
print("combined", comb_cpg.shape)
